
# DATA 304 — Demo: Advanced Text Representations

This notebook demonstrates:
- Latent Semantic Analysis (LSA)
- Latent Dirichlet Allocation (LDA)
- Word embeddings (Word2Vec)
- Contextual sentence embeddings (BERT via Sentence Transformers)
- Applying different representations in a simple sentiment pipeline


## 1. Setup

In [ ]:

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
# %matplotlib inline


## 2. Toy Corpus for Topics / Concepts

In [ ]:

docs = [
    "Data wrangling with pandas and Python for messy datasets",
    "Cleaning text data and removing noise is essential",
    "Machine learning models use numeric features like TF IDF",
    "Topic models like LDA discover themes in document collections",
    "LSA reduces TF IDF into latent concepts for similarity search",
    "Word embeddings capture semantic similarity between words",
    "BERT provides contextual sentence embeddings for NLP tasks",
    "Clustering documents by topic helps organize large corpora"
]

len(docs), docs[:3]


## 3. TF-IDF Representation (Baseline)

In [ ]:

tfidf = TfidfVectorizer(stop_words='english')
X_tfidf = tfidf.fit_transform(docs)

tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out()).round(3)
tfidf_df.head()


## 4. Latent Semantic Analysis (LSA)

In [ ]:

# Reduce TF-IDF to a small number of latent concepts
svd = TruncatedSVD(n_components=2, random_state=42)
X_lsa = svd.fit_transform(X_tfidf)

lsa_df = pd.DataFrame(X_lsa, columns=["Concept1", "Concept2"])
lsa_df


In [ ]:

# Inspect top terms for each latent concept
terms = tfidf.get_feature_names_out()
n_top = 8

for i, comp in enumerate(svd.components_):
    top_ids = comp.argsort()[-n_top:][::-1]
    top_terms = [terms[j] for j in top_ids]
    print(f"Concept {i+1}: {', '.join(top_terms)}")


## 5. Latent Dirichlet Allocation (LDA)

In [ ]:

# LDA works on count (BoW) matrix, not TF-IDF
vec = CountVectorizer(max_df=0.95, min_df=1, stop_words='english')
X_counts = vec.fit_transform(docs)

lda = LatentDirichletAllocation(n_components=3, random_state=42)
lda.fit(X_counts)


In [ ]:

# View top words per topic
terms = vec.get_feature_names_out()
n_top = 8

for i, comp in enumerate(lda.components_):
    top_ids = comp.argsort()[-n_top:][::-1]
    top_terms = [terms[j] for j in top_ids]
    print(f"Topic {i+1}: {', '.join(top_terms)}")


In [ ]:

# Document–topic distributions
doc_topic = lda.transform(X_counts)
pd.DataFrame(doc_topic.round(3),
             columns=[f"Topic{i+1}" for i in range(lda.n_components)])


## 6. Word Embeddings (Word2Vec)

In [ ]:

# Simple tokenization for demo purposes
tokenized_texts = [doc.lower().split() for doc in docs]
tokenized_texts[:2]


In [ ]:

from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=tokenized_texts,
    vector_size=100,
    window=5,
    min_count=1,
    workers=2,
    sg=1  # use skip-gram
)

w2v_model.wv.most_similar("data")


In [ ]:

# Inspect similarity between some technical terms if present
for pair in [("data", "pandas"),
             ("lda", "topic"),
             ("bert", "embeddings")]:
    w1, w2 = pair
    if w1 in w2v_model.wv.key_to_index and w2 in w2v_model.wv.key_to_index:
        print(f"cosine_sim({w1}, {w2}) = {w2v_model.wv.similarity(w1, w2):.3f}")


## 7. Contextual Sentence Embeddings (BERT via Sentence Transformers)

In [ ]:

# This cell requires the sentence-transformers package to be installed.
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "He sat by the bank of the river.",
    "He deposited money in the bank.",
    "Data wrangling with pandas is powerful."
]

embeddings = st_model.encode(sentences)
embeddings.shape


In [ ]:

# Compare similarity between different uses of 'bank'
sim_matrix = cosine_similarity(embeddings)
pd.DataFrame(sim_matrix.round(3),
             index=[f"Sent{i+1}" for i in range(len(sentences))],
             columns=[f"Sent{i+1}" for i in range(len(sentences))])


## 8. Simple Sentiment Pipeline with Different Representations

### 8.1 Toy sentiment dataset

In [ ]:
import nltk
nltk.download("movie_reviews")

from nltk.corpus import movie_reviews
import numpy as np

texts = [" ".join(movie_reviews.words(fileid)) 
         for fileid in movie_reviews.fileids()]

y = np.array([1 if fileid.startswith("pos") else 0 
              for fileid in movie_reviews.fileids()])


### 8.2 TF-IDF + Logistic Regression (baseline)

In [ ]:

tfidf_sent = TfidfVectorizer(stop_words='english')
X_tfidf_sent = tfidf_sent.fit_transform(texts)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf_sent, y, test_size=0.25, random_state=42
)

clf_tfidf = LogisticRegression(max_iter=1000)
clf_tfidf.fit(X_train, y_train)

print("TF-IDF accuracy:", clf_tfidf.score(X_test, y_test))


### 8.3 LSA (reduced concepts) + Logistic Regression

In [ ]:

svd_sent = TruncatedSVD(n_components=2, random_state=42)
X_lsa_sent = svd_sent.fit_transform(X_tfidf_sent)

X_train_lsa, X_test_lsa, y_train_lsa, y_test_lsa = train_test_split(
    X_lsa_sent, y, test_size=0.25, random_state=42
)

clf_lsa = LogisticRegression(max_iter=1000)
clf_lsa.fit(X_train_lsa, y_train_lsa)

print("LSA accuracy:", clf_lsa.score(X_test_lsa, y_test_lsa))


### 8.4 Word2Vec document embeddings + Logistic Regression

In [ ]:

# Train a small Word2Vec on sentiment texts
sent_tokenized = [t.lower().split() for t in texts]

w2v_sent_model = Word2Vec(
    sentences=sent_tokenized,
    vector_size=50,
    window=5,
    min_count=1,
    workers=2,
    sg=1
)

def doc_vector(tokens, model):
    vecs = [model.wv[w] for w in tokens if w in model.wv.key_to_index]
    if not vecs:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

X_w2v_sent = np.vstack([doc_vector(tokens, w2v_sent_model)
                        for tokens in sent_tokenized])

X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = train_test_split(
    X_w2v_sent, y, test_size=0.25, random_state=42
)

clf_w2v = LogisticRegression(max_iter=1000)
clf_w2v.fit(X_train_w2v, y_train_w2v)

print("Word2Vec accuracy:", clf_w2v.score(X_test_w2v, y_test_w2v))


### 8.5 BERT embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

# 1) Build sentence embeddings with a small, fast BERT model
bert_model = SentenceTransformer('all-MiniLM-L6-v2')

X_bert = bert_model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

# 2) Train/test split (same pattern as before)
X_train_bert, X_test_bert, y_train_bert, y_test_bert = train_test_split(
    X_bert,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3) Train Logistic Regression on BERT embeddings
clf_bert = LogisticRegression(max_iter=2000)
clf_bert.fit(X_train_bert, y_train_bert)

print("BERT accuracy:", clf_bert.score(X_test_bert, y_test_bert))
